<a href="https://colab.research.google.com/github/beefann236/datast_penjualan2/blob/main/datast_penjualan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [48]:
import pandas as pd
# DataFrame - seperti tabel Excel di dalam Python
df = pd.read_csv("/content/dataset_penjualan_kantin - dataset_penjualan_kantin.csv")
df.head() # lihat 5 baris pertama
df.info() # lihat tipe data & jumlah data tiap kolom
df.describe() # lihat statistik ringkas (rata-rata, min, max, dst)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69 entries, 0 to 68
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_transaksi       69 non-null     object
 1   tanggal            69 non-null     object
 2   nama_produk        69 non-null     object
 3   kategori           69 non-null     object
 4   jumlah_terjual     65 non-null     object
 5   harga_satuan       66 non-null     object
 6   nama_kasir         66 non-null     object
 7   metode_pembayaran  69 non-null     object
dtypes: object(8)
memory usage: 4.4+ KB


,id_transaksi,tanggal,nama_produk,kategori,jumlah_terjual,harga_satuan,nama_kasir,metode_pembayaran
count,69,69,69,69,65,66,66,69
unique,65,27,12,9,21,16,4,3
top,TRX0042,2026-08-11,Jus Alpukat,Makanan,9,10000,Pak Joko,QRIS
freq,2,7,11,19,12,10,22,36


### 2. Data Cleaning

Based on the initial inspection, we need to address missing values, ensure correct data types, and handle potential duplicates. We will document the reasons for each cleaning technique.

In [64]:
# Memuat ulang DataFrame proyek utama untuk memastikan operasi cleaning dan manipulasi berjalan pada dataset yang benar.
df = pd.read_csv("/content/dataset_penjualan_kantin - dataset_penjualan_kantin.csv")
print("DataFrame proyek utama berhasil dimuat ulang.")

DataFrame proyek utama berhasil dimuat ulang.


In [65]:
# Identify missing values
print("Jumlah Missing Values Sebelum Cleaning:")
print(df.isnull().sum())

Jumlah Missing Values Sebelum Cleaning:
id_transaksi         0
tanggal              0
nama_produk          0
kategori             0
jumlah_terjual       4
harga_satuan         3
nama_kasir           3
metode_pembayaran    0
dtype: int64


In [74]:
print(f"Columns at start of cb1f5b53: {df.columns.tolist()}")

# Handle 'jumlah_terjual' (object) and 'harga_satuan' (object)
# Convert to numeric, coercing errors to NaN, then fill NaN with 0
# Alasan: Kolom ini seharusnya numerik untuk perhitungan. Mengisi dengan 0 berasumsi bahwa nilai yang hilang berarti tidak ada penjualan atau harga gratis, yang masuk akal untuk data penjualan kantin.
df['jumlah_terjual'] = pd.to_numeric(df['jumlah_terjual'], errors='coerce').fillna(0).astype(int)
df['harga_satuan'] = pd.to_numeric(df['harga_satuan'], errors='coerce').fillna(0).astype(int)

# Handle 'nama_kasir' (object)
# Alasan: Mengisi dengan 'Unknown' untuk mempertahankan baris data yang lain namun menunjukkan informasi kasir tidak tersedia.
df['nama_kasir'] = df['nama_kasir'].fillna('Unknown')

# Convert 'tanggal' to datetime type
# Alasan: Untuk memungkinkan analisis berbasis waktu dan sorting yang benar.
# Mengganti nama bulan dalam Bahasa Indonesia ke Bahasa Inggris untuk parsing yang benar
indonesian_months = {
    'Januari': 'January', 'Februari': 'February', 'Maret': 'March',
    'April': 'April', 'Mei': 'May', 'Juni': 'June', 'Juli': 'July',
    'Agustus': 'August', 'September': 'September', 'Oktober': 'October',
    'November': 'November', 'Desember': 'December'
}
def replace_indonesian_month(date_str):
    if not isinstance(date_str, str): # Handle non-string values like NaN
        return date_str
    for indo, eng in indonesian_months.items():
        date_str = date_str.replace(indo, eng)
    return date_str

df['tanggal'] = df['tanggal'].apply(replace_indonesian_month)
df['tanggal'] = pd.to_datetime(df['tanggal'], format='mixed', dayfirst=True)

print("Jumlah Missing Values Setelah Cleaning:")
print(df.isnull().sum())
print("\nData Types Setelah Cleaning:")
print(df.dtypes)

Columns at start of cb1f5b53: ['id_transaksi', 'tanggal', 'nama_produk', 'kategori', 'jumlah_terjual', 'harga_satuan', 'nama_kasir', 'metode_pembayaran']
Jumlah Missing Values Setelah Cleaning:
id_transaksi         0
tanggal              0
nama_produk          0
kategori             0
jumlah_terjual       0
harga_satuan         0
nama_kasir           0
metode_pembayaran    0
dtype: int64

Data Types Setelah Cleaning:
id_transaksi                 object
tanggal              datetime64[ns]
nama_produk                  object
kategori                     object
jumlah_terjual                int64
harga_satuan                  int64
nama_kasir                   object
metode_pembayaran            object
dtype: object


In [59]:
# Handle duplicate data
print(f"Jumlah duplikat sebelum drop: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Jumlah duplikat setelah drop: {df.duplicated().sum()}")

Jumlah duplikat sebelum drop: 0
Jumlah duplikat setelah drop: 0


### 3. Data Manipulation

We will now perform filtering, sorting, create a derived column, and aggregate data as required by the project charter.

In [75]:
# Create derived column: 'total_pendapatan'
# Alasan: Untuk menghitung total pendapatan dari setiap transaksi.
df['total_pendapatan'] = df['jumlah_terjual'] * df['harga_satuan']
print("DataFrame dengan Kolom 'total_pendapatan':")
display(df.head())

DataFrame dengan Kolom 'total_pendapatan':


,id_transaksi,tanggal,nama_produk,kategori,jumlah_terjual,harga_satuan,nama_kasir,metode_pembayaran,total_pendapatan
0,TRX0042,2026-08-11,Roti Bakar,Makanan,9,7000,Pak Agus,Tunai,63000
1,TRX0005,2026-08-03,Gorengan,makanan,2,2000,Bu Sri,QRIS,4000
2,TRX0011,2026-08-04,Jus Alpukat,Minuman,4,0,Bu Wati,Tunai,0
3,TRX0035,2026-08-10,Mie Ayam,Makanan,0,10000,Pak Joko,Transfer,0
4,TRX0007,2026-08-03,Kerupuk,Snack,10,0,Bu Wati,Transfer,0


In [76]:
# Filtering: Filter transaksi dengan 'total_pendapatan' di atas rata-rata
# Alasan: Untuk mengidentifikasi transaksi dengan nilai tinggi yang mungkin menunjukkan penjualan produk populer atau pembeli besar.
rata_rata_pendapatan = df['total_pendapatan'].mean()
df_high_revenue = df[df['total_pendapatan'] > rata_rata_pendapatan]
print(f"Transaksi dengan total pendapatan di atas rata-rata ({rata_rata_pendapatan:.2f}):")
display(df_high_revenue.head())

Transaksi dengan total pendapatan di atas rata-rata (85376.81):


,id_transaksi,tanggal,nama_produk,kategori,jumlah_terjual,harga_satuan,nama_kasir,metode_pembayaran,total_pendapatan
7,TRX0008,2026-08-04,Mie Ayam,makanan,14,10000,Pak Joko,Tunai,140000
9,TRX0057,2026-08-13,Jus Alpukat,MINUMAN,14,8000,Bu Sri,Tunai,112000
16,TRX0054,2026-08-13,Jus Alpukat,Minuman,15,8000,Pak Joko,Tunai,120000
20,TRX0046,2026-08-11,Nasi Goreng,Makanan,15,12000,Bu Wati,QRIS,180000
21,TRX0003,2026-08-03,Nasi Goreng,MAKANAN,9,12000,Pak Joko,Transfer,108000


In [77]:
# Sorting: Urutkan data berdasarkan 'total_pendapatan' secara menurun
# Alasan: Untuk dengan mudah melihat transaksi mana yang paling menguntungkan.
df_sorted_revenue = df.sort_values(by='total_pendapatan', ascending=False)
print("Data diurutkan berdasarkan Total Pendapatan:")
display(df_sorted_revenue.head())

Data diurutkan berdasarkan Total Pendapatan:


,id_transaksi,tanggal,nama_produk,kategori,jumlah_terjual,harga_satuan,nama_kasir,metode_pembayaran,total_pendapatan
59,TRX0050,2026-08-12,Roti Bakar,MAKANAN,500,7000,Pak Joko,QRIS,3500000
20,TRX0046,2026-08-11,Nasi Goreng,Makanan,15,12000,Bu Wati,QRIS,180000
7,TRX0008,2026-08-04,Mie Ayam,makanan,14,10000,Pak Joko,Tunai,140000
26,TRX0059,2026-08-14,Bakso,Makanan,11,11000,Bu Sri,Tunai,121000
23,TRX0019,2026-08-06,Nasi Goreng,Makanan,10,12000,Bu Wati,QRIS,120000


In [78]:
# Groupby/Agregasi: Hitung total pendapatan per 'nama_produk'
# Alasan: Untuk menganalisis kontribusi pendapatan dari setiap produk dan mengidentifikasi produk terlaris.
ringkasan_pendapatan_produk = df.groupby('nama_produk')['total_pendapatan'].sum().sort_values(ascending=False)
print("Total Pendapatan per Nama Produk:")
display(ringkasan_pendapatan_produk)

Total Pendapatan per Nama Produk:


,total_pendapatan
nama_produk,
Roti Bakar,3675000
Nasi Goreng,540000
Jus Alpukat,480000
Mie Ayam,420000
Bakso,253000
Nasi Uduk,168000
Teh Botol,145000
Es Teh,81000
Kerupuk,62000


In [79]:
# Simpan DataFrame yang telah dibersihkan ke file CSV baru
# Alasan: Untuk menyediakan dataset yang siap digunakan untuk analisis lebih lanjut atau visualisasi.
df.to_csv('dataset_bersih.csv', index=False)
print("DataFrame yang telah dibersihkan berhasil disimpan ke 'dataset_bersih.csv'")

DataFrame yang telah dibersihkan berhasil disimpan ke 'dataset_bersih.csv'


In [80]:
# Create derived column: 'total_pendapatan'
# Alasan: Untuk menghitung total pendapatan dari setiap transaksi.
df['total_pendapatan'] = df['jumlah_terjual'] * df['harga_satuan']
print("DataFrame dengan Kolom 'total_pendapatan':")
display(df.head())

DataFrame dengan Kolom 'total_pendapatan':


,id_transaksi,tanggal,nama_produk,kategori,jumlah_terjual,harga_satuan,nama_kasir,metode_pembayaran,total_pendapatan
0,TRX0042,2026-08-11,Roti Bakar,Makanan,9,7000,Pak Agus,Tunai,63000
1,TRX0005,2026-08-03,Gorengan,makanan,2,2000,Bu Sri,QRIS,4000
2,TRX0011,2026-08-04,Jus Alpukat,Minuman,4,0,Bu Wati,Tunai,0
3,TRX0035,2026-08-10,Mie Ayam,Makanan,0,10000,Pak Joko,Transfer,0
4,TRX0007,2026-08-03,Kerupuk,Snack,10,0,Bu Wati,Transfer,0


In [61]:
# Filtering: Filter transaksi dengan 'total_pendapatan' di atas rata-rata
# Alasan: Untuk mengidentifikasi transaksi dengan nilai tinggi yang mungkin menunjukkan penjualan produk populer atau pembeli besar.
rata_rata_pendapatan = df['total_pendapatan'].mean()
df_high_revenue = df[df['total_pendapatan'] > rata_rata_pendapatan]
print(f"Transaksi dengan total pendapatan di atas rata-rata ({rata_rata_pendapatan:.2f}):")
display(df_high_revenue.head())

Transaksi dengan total pendapatan di atas rata-rata (144000.00):


,menu,harga,terjual,total_pendapatan
0,Nasi Goreng,12000,23.0,276000.0
1,Es Teh,4000,40.0,160000.0


In [62]:
# Sorting: Urutkan data berdasarkan 'total_pendapatan' secara menurun
# Alasan: Untuk dengan mudah melihat transaksi mana yang paling menguntungkan.
df_sorted_revenue = df.sort_values(by='total_pendapatan', ascending=False)
print("Data diurutkan berdasarkan Total Pendapatan:")
display(df_sorted_revenue.head())

Data diurutkan berdasarkan Total Pendapatan:


,menu,harga,terjual,total_pendapatan
0,Nasi Goreng,12000,23.0,276000.0
1,Es Teh,4000,40.0,160000.0
3,Es Teh,4000,35.0,140000.0
2,Mie Ayam,10000,0.0,0.0


In [49]:
import numpy as np
nilai = np.array([80, 75, 90, 60, 88])
print(nilai.mean()) # rata-rata
print(nilai.max()) # nilai tertinggi
print(nilai * 2) # operasi langsung ke semua elemen (vectorized)


78.6
90
[160 150 180 120 176]


In [67]:
import pandas as pd
data = {
 'nama': ['Andi', 'Budi', 'Citra'],
 'nilai': [80, 75, 90]
}
df_names = pd.DataFrame(data) # Mengubah nama variabel dari df menjadi df_names
print(df_names)

    nama  nilai
0   Andi     80
1   Budi     75
2  Citra     90


In [51]:
import numpy as np
harga = np.array([5000, 7000, 3000, 12000, 4500])
print('Rata-rata harga:', harga.mean())
print('Harga tertinggi:', harga.max())
print('Harga setelah diskon 10%:', harga * 0.9)


Rata-rata harga: 6300.0
Harga tertinggi: 12000
Harga setelah diskon 10%: [ 4500.  6300.  2700. 10800.  4050.]


In [66]:
import pandas as pd
data_kantin = {
 'menu': ['Nasi Goreng', 'Es Teh', 'Mie Ayam', 'Es Teh', None],
 'harga': [12000, 4000, 10000, 4000, 8000],
 'terjual': [23, 40, None, 35, 18]
}
df_example = pd.DataFrame(data_kantin) # Mengubah nama variabel dari df menjadi df_example
print(df_example)

          menu  harga  terjual
0  Nasi Goreng  12000     23.0
1       Es Teh   4000     40.0
2     Mie Ayam  10000      NaN
3       Es Teh   4000     35.0
4         None   8000     18.0


In [68]:
print(df_example.head()) # 5 baris pertama
print(df_example.info()) # tipe data & jumlah non-null tiap kolom
print(df_example.describe()) # statistik ringkas kolom numerik
print(df_example.shape) # jumlah (baris, kolom)

          menu  harga  terjual
0  Nasi Goreng  12000     23.0
1       Es Teh   4000     40.0
2     Mie Ayam  10000      NaN
3       Es Teh   4000     35.0
4         None   8000     18.0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   menu     4 non-null      object 
 1   harga    5 non-null      int64  
 2   terjual  4 non-null      float64
dtypes: float64(1), int64(1), object(1)
memory usage: 252.0+ bytes
None
              harga    terjual
count      5.000000   4.000000
mean    7600.000000  29.000000
std     3577.708764  10.230673
min     4000.000000  18.000000
25%     4000.000000  21.750000
50%     8000.000000  29.000000
75%    10000.000000  36.250000
max    12000.000000  40.000000
(5, 3)


In [69]:
print(df_example.isnull().sum()) # jumlah data kosong tiap kolom
df_example['terjual'] = df_example['terjual'].fillna(0) # isi kekosongan dengan 0
df_example = df_example.dropna(subset=['menu']) # hapus baris jika kolom menu kosong

menu       1
harga      0
terjual    1
dtype: int64


In [70]:
print(df_example.duplicated().sum()) # jumlah baris duplikat
df_example = df_example.drop_duplicates()
df_example['harga'] = df_example['harga'].astype(int) # memastikan tipe data harga adalah integer
print(df_example.dtypes)

0
menu        object
harga        int64
terjual    float64
dtype: object


In [71]:
laris = df_example[df_example['terjual'] > 20] # filtering
urut = df_example.sort_values(by='terjual', ascending=False) # sorting
df_example['total_pendapatan'] = df_example['harga'] * df_example['terjual'] # kolom turunan
ringkasan = df_example.groupby('menu')['total_pendapatan'].sum() # agregasi
print(ringkasan)

menu
Es Teh         300000.0
Mie Ayam            0.0
Nasi Goreng    276000.0
Name: total_pendapatan, dtype: float64
